In [17]:
#pip install openpyxl

In [1]:
import networkx as nx
import numpy as np
from collections import Counter
from gwgraphs import *

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("BZR_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("BZR_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("BZR_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("BZR_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("BZR_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float,
                     line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def read_family_labels():
  # Read family labels from the Excel file
  df = pd.read_excel("BZR.xlsx")
  # Extract the "Fam." column and convert it to a list
  family_labels = df["Fam"].tolist()
  return family_labels

In [3]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data(
)
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
    )

Graph ID: 1, number of nodes: 30, label: -1
Graph ID: 2, number of nodes: 33, label: -1
Graph ID: 3, number of nodes: 30, label: -1
Graph ID: 4, number of nodes: 33, label: -1
Graph ID: 5, number of nodes: 30, label: -1
Graph ID: 6, number of nodes: 33, label: -1
Graph ID: 7, number of nodes: 30, label: -1
Graph ID: 8, number of nodes: 30, label: -1
Graph ID: 9, number of nodes: 33, label: -1
Graph ID: 10, number of nodes: 30, label: -1
Graph ID: 11, number of nodes: 30, label: -1
Graph ID: 12, number of nodes: 30, label: -1
Graph ID: 13, number of nodes: 33, label: -1
Graph ID: 14, number of nodes: 33, label: -1
Graph ID: 15, number of nodes: 33, label: -1
Graph ID: 16, number of nodes: 33, label: -1
Graph ID: 17, number of nodes: 32, label: -1
Graph ID: 18, number of nodes: 32, label: -1
Graph ID: 19, number of nodes: 32, label: -1
Graph ID: 20, number of nodes: 35, label: -1
Graph ID: 21, number of nodes: 32, label: -1
Graph ID: 22, number of nodes: 35, label: -1
Graph ID: 23, numbe

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]
# Determine number of elements of each class
print("Elements of Class 1: ", labels_list.count(1))
print("Elements of Class -1: ", labels_list.count(-1))

Elements of Class 1:  86
Elements of Class -1:  319


In [5]:
## BZR Ligand Family Information ##
family_labels = read_family_labels()
#print("Family Labels:", family_labels)

In [6]:
# Count occurrences of each family label
family_counts = Counter(family_labels)
print("Original family label counts:", dict(family_counts))

# Keep only families with 10 or more elements
valid_families = {family for family, count in family_counts.items() if count >= 10}
print("Families with 10+ elements:", valid_families)

# Create indices of graphs to keep (where family label is in valid_families)
valid_indices = [i for i, family in enumerate(family_labels) if family in valid_families]

# Filter family labels
filtered_family_labels = [family_labels[i] for i in valid_indices]

# Create filtered graph lists
graphs_list_fam = [graphs_list[i] for i in valid_indices]
graphs_list_fam_normalized = [list(graphs_normalized.values())[i] for i in valid_indices]

# Print remaining family labels with counts
remaining_counts = Counter(filtered_family_labels)
print("Remaining family labels with counts:")
for family, count in remaining_counts.items():
    print(f"  Family {family}: {count} elements")

Original family label counts: {'A.1': 91, 'A.2': 3, 'A.3': 1, 'A.4': 1, 'A.5': 5, 'A.6': 1, 'A.7': 1, 'A.8': 1, 'A.9': 2, 'A.10': 1, 'A.11': 4, 'A.12': 2, 'A.13': 10, 'A.14': 6, 'A.15': 10, 'A.16': 1, 'A.17': 2, 'A.18': 6, 'A.19': 1, 'A.20': 6, 'A.21': 1, 'A.22': 4, 'A.23': 5, 'A.24': 32, 'A.25': 9, 'A.26': 10, 'A.27': 1, 'B': 1, 'C': 1, 'D': 1, 'E.1': 3, 'E.2': 10, 'F': 5, 'G.1': 3, 'G.2': 6, 'H': 3, 'I': 28, 'J': 3, 'K.1': 3, 'K.2': 2, 'L.1': 2, 'L.2': 2, 'L.3': 1, 'M.1': 45, 'M.2': 1, 'M.3': 1, 'M.4': 5, 'M.5': 15, 'N.1': 40, 'N.2': 1, 'N.3': 1, 'N.4': 3, 'N.5': 1, 'N.6': 1}
Families with 10+ elements: {'A.26', 'I', 'M.5', 'A.13', 'A.24', 'N.1', 'M.1', 'E.2', 'A.15', 'A.1'}
Remaining family labels with counts:
  Family A.1: 91 elements
  Family A.13: 10 elements
  Family A.15: 10 elements
  Family A.24: 32 elements
  Family A.26: 10 elements
  Family E.2: 10 elements
  Family I: 28 elements
  Family M.1: 45 elements
  Family M.5: 15 elements
  Family N.1: 40 elements


In [7]:
distance_matrices_fam = [compute_bzr_distance_matrix(graph) for graph in graphs_list_fam]

In [8]:
"""GCN architeture for multiclass classification"""
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np
import time
import random
import optuna
from optuna.samplers import TPESampler
from gwgraphs import *
from visualizations import GCNResultsVisualizer, get_model_probabilities

In [9]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [10]:
class GraphNeuralNetwork(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=3):
        super(GraphNeuralNetwork, self).__init__()

        self.num_layers = num_layers
        self.convs = nn.ModuleList()

        # First layer
        self.convs.append(GCNConv(input_dim, hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))

        # Last layer
        self.convs.append(GCNConv(hidden_dim, hidden_dim))

        # Classifier
        self.classifier = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, edge_index, batch):
        # Apply GCN layers
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = self.dropout(x)

        # Global pooling to get graph-level representation
        x = global_mean_pool(x, batch)

        # Final classification
        x = self.classifier(x)

        return x


def train_model_multi(model, train_loader, optimizer, criterion, device, label_to_idx):
    """Function to train the model for one epoch - supports multi-class."""
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Forward pass
        out = model(batch.x, batch.edge_index, batch.batch)

        # Convert labels to indices for cross-entropy loss
        labels_idx = torch.tensor([label_to_idx[label.item()] for label in batch.y], device=device)

        loss = criterion(out, labels_idx)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate_model_multi(model, test_loader, device, idx_to_label):
    """Function to evaluate the model - supports multi-class."""
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)

            # Get predictions
            pred_idx = torch.argmax(out, dim=1)
            # Convert back to original labels
            pred_labels = [idx_to_label[idx.item()] for idx in pred_idx]

            predictions.extend(pred_labels)
            true_labels.extend([label.item() for label in batch.y])

        return np.array(predictions), np.array(true_labels)


def calculate_metrics_multi(y_true, y_pred, unique_classes):
    """Calculate classification metrics for multi-class problems"""
    # Overall accuracy
    overall_acc = accuracy_score(y_true, y_pred) * 100

    # By-class accuracy
    class_accuracies = {}
    for class_label in unique_classes:
        class_mask = y_true == class_label
        if np.sum(class_mask) > 0:
            class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask]) * 100
            class_accuracies[class_label] = class_acc
        else:
            class_accuracies[class_label] = 0.0

    # F1 score
    f1 = f1_score(y_true, y_pred, average='weighted') * 100

    return overall_acc, class_accuracies, f1


def objective_multi(trial, full_graph_data, labels, device, input_dim, hidden_dim, output_dim, unique_classes):
    """Objective function for Bayesian optimization - supports multi-class"""
    # Suggest hyperparameters
    num_epochs = trial.suggest_int('num_epochs', 20, 100, step=10)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    # Create label mappings
    label_to_idx = {label: idx for idx, label in enumerate(sorted(unique_classes))}
    idx_to_label = {idx: label for label, idx in label_to_idx.items()}

    # Cross validation for this trial
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for fold, (train_idx, test_idx) in enumerate(kfold.split(full_graph_data, labels)):
        # Split data
        train_graphs = [full_graph_data[i] for i in train_idx]
        test_graphs = [full_graph_data[i] for i in test_idx]

        # Create data loaders
        train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

        # Initialize model
        model = GraphNeuralNetwork(input_dim, hidden_dim, output_dim).to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()

        # Training loop
        for epoch in range(num_epochs):
             train_model_multi(model, train_loader, optimizer, criterion, device, label_to_idx)

        # Evaluation
        y_pred, y_true = evaluate_model_multi(model, test_loader, device, idx_to_label)
        overall_acc, _, _ = calculate_metrics_multi(y_true, y_pred, unique_classes)
        fold_scores.append(overall_acc)

        # Report intermediate objective value for pruning
        trial.report(np.mean(fold_scores[:fold+1]), fold)

        # Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_scores)


def networkx_to_pyg(nx_graph, label, num_features):
    """Convert NetworkX graph to PyTorch Geometric Data object"""
    num_nodes = len(nx_graph.nodes)

    # Extract node features from the graph's node attributes
    node_features = []
    node_list = sorted(nx_graph.nodes())  # Ensure consistent ordering

    for node in node_list:
        if 'attributes' in nx_graph.nodes[node]:
            # Use the actual node attributes from the BZR dataset
            node_features.append(nx_graph.nodes[node]['attributes'])
        else:
            # Fallback to zero features if attributes are missing
            node_features.append([0.0] * num_features)

    # Convert to tensor
    x = torch.tensor(node_features, dtype=torch.float32)

    # Convert edges to tensor format - need to map node IDs to indices
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}
    edge_list = list(nx_graph.edges())

    if len(edge_list) == 0:
        # Handle graphs with no edges
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        # Map node IDs to indices for edge_index
        mapped_edges = [(node_to_idx[u], node_to_idx[v]) for u, v in edge_list]
        edge_index = torch.tensor(mapped_edges, dtype=torch.long).t().contiguous()

    # Convert label to tensor
    y = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, y=y)


def gcn_multi_class_classification(graph_data_list, labels_list, input_dim, hidden_dim=64, num_folds=5, seed=42):
    """
    Perform multi-class classification using GCN with comprehensive metrics similar to 
    gw_knn_multi and fgw_knn_alpha_multi functions.

    Parameters:
    - graph_data_list: list of PyTorch Geometric Data objects
    - labels_list: list of labels for each graph
    - input_dim: number of input features
    - hidden_dim: number of hidden dimensions
    - num_folds: number of folds for cross-validation
    - seed: random seed for reproducibility

    Returns:
    - class_acc_dict: dictionary storing accuracies for each fold
    - by_class_acc_dict: dictionary storing accuracies by class for each fold  
    - f1_score_dict: dictionary storing F1 scores for each fold
    - best_results_info: dictionary storing final averaged results
    """

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Get unique classes from the labels
    unique_classes = sorted(list(set(labels_list)))
    output_dim = len(unique_classes)
    print(f"Multi-class classification with {len(unique_classes)} classes: {unique_classes}")

    # Create label mappings
    label_to_idx = {label: idx for idx, label in enumerate(unique_classes)}
    idx_to_label = {idx: label for label, idx in label_to_idx.items()}

    # Initialize result dictionaries
    class_acc_dict = {i: [] for i in range(num_folds)}
    by_class_acc_dict = {i: {class_label: [] for class_label in unique_classes} for i in range(num_folds)}
    f1_score_dict = {i: [] for i in range(num_folds)}

    print("\nStarting Bayesian Optimization for Hyperparameter Tuning...")
    start_time = time.time()

    # Create Optuna study
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
    )

    # Optimize hyperparameters
    study.optimize(
        lambda trial: objective_multi(trial, graph_data_list, labels_list, device, input_dim, hidden_dim, output_dim, unique_classes),
        n_trials=50 
    )

    optimization_time = time.time() - start_time

    # Print best hyperparameters
    print("\n" + "=" * 60)
    print("BAYESIAN OPTIMIZATION RESULTS")
    print("=" * 60)
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best accuracy: {study.best_trial.value:.2f}%")
    print("Best hyperparameters:")
    for key, value in study.best_trial.params.items():
        print(f"  {key}: {value}")

    print(f"\nOptimization time: {optimization_time:.2f} seconds")

    # Train final model with best hyperparameters
    print("\n" + "=" * 60)
    print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
    print("=" * 60)

    best_params = study.best_trial.params
    final_num_epochs = best_params['num_epochs']
    final_learning_rate = best_params['learning_rate']
    final_batch_size = best_params['batch_size']

    # Cross validation with best parameters
    kfold = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
    all_results = []

    for fold, (train_idx, test_idx) in enumerate(kfold.split(graph_data_list, labels_list)):
        print(f"\n--- Training Fold {fold + 1}/{num_folds} ---")

        # Split data
        train_graphs = [graph_data_list[i] for i in train_idx]
        test_graphs = [graph_data_list[i] for i in test_idx]

        # Create data loaders
        train_loader = DataLoader(train_graphs, batch_size=final_batch_size, shuffle=True)
        test_loader = DataLoader(test_graphs, batch_size=final_batch_size, shuffle=False)

        # Initialize model
        model = GraphNeuralNetwork(input_dim, hidden_dim, output_dim).to(device)
        optimizer = optim.Adam(model.parameters(), lr=final_learning_rate)
        criterion = nn.CrossEntropyLoss()

        # Training loop
        for epoch in range(final_num_epochs):
            train_loss = train_model_multi(model, train_loader, optimizer, criterion, device, label_to_idx)

            if (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch + 1}/{final_num_epochs}, Loss: {train_loss:.4f}")

        # Final evaluation
        y_pred, y_true = evaluate_model_multi(model, test_loader, device, idx_to_label)
        overall_acc, class_accuracies, f1 = calculate_metrics_multi(y_true, y_pred, unique_classes)

        # Store results
        class_acc_dict[fold].append(overall_acc)
        f1_score_dict[fold].append(f1)

        for class_label in unique_classes:
            by_class_acc_dict[fold][class_label].append(class_accuracies[class_label])

        results = {
            'fold': fold + 1,
            'overall_accuracy': overall_acc,
            'f1_score': f1
        }

        # Add class-specific accuracies
        for class_label in unique_classes:
            results[f'class_{class_label}_accuracy'] = class_accuracies[class_label]

        all_results.append(results)

        print(f"Results for Fold {fold + 1}:")
        print(f"  Overall Accuracy: {overall_acc:.2f}%")
        print(f"  F1 Score (weighted avg): {f1:.2f}%")
        for class_label in unique_classes:
            print(f"  Class {class_label} Accuracy: {class_accuracies[class_label]:.2f}%")

    # Calculate final average metrics and standard deviations
    all_overall_acc = [acc for fold_accs in class_acc_dict.values() for acc in fold_accs]
    all_f1_scores = [f1 for fold_f1s in f1_score_dict.values() for f1 in fold_f1s]

    avg_overall_acc = np.mean(all_overall_acc)
    avg_f1 = np.mean(all_f1_scores)

    std_overall_acc = np.std(all_overall_acc)
    std_f1 = np.std(all_f1_scores)

    # Calculate class-specific averages and standard deviations
    class_averages = {}
    class_stds = {}

    for class_label in unique_classes:
        all_class_accs = [acc for fold_dict in by_class_acc_dict.values() for acc in fold_dict[class_label]]
        class_averages[class_label] = np.mean(all_class_accs)
        class_stds[class_label] = np.std(all_class_accs)

    total_time = time.time() - start_time

    print("\n" + "=" * 60)
    print("FINAL GCN MULTI-CLASS RESULTS")
    print("=" * 60)
    print(f"Best hyperparameters used:")
    print(f"  Epochs: {final_num_epochs}")
    print(f"  Learning Rate: {final_learning_rate:.6f}")
    print(f"  Batch Size: {final_batch_size}")
    print(f"\nFinal Cross-Validation Results:")
    print(f"Average Overall Accuracy: {avg_overall_acc:.2f}% ± {std_overall_acc:.2f}%")
    print(f"Average F1 Score (weighted avg): {avg_f1:.2f}% ± {std_f1:.2f}%")

    for class_label in unique_classes:
        print(f"Average Class {class_label} Accuracy: {class_averages[class_label]:.2f}% ± {class_stds[class_label]:.2f}%")

    print(f"\nTotal Time (Optimization + Training): {total_time:.2f} seconds")

    # Prepare results info similar to other functions
    best_results_info = {
        'overall_accuracy': avg_overall_acc,
        'overall_accuracy_std': std_overall_acc,
        'f1_score': avg_f1,
        'f1_score_std': std_f1,
        'hyperparameters': best_params,
        'total_time': total_time
    }

    # Add class-specific results
    for class_label in unique_classes:
        best_results_info[f'class_{class_label}_accuracy'] = class_averages[class_label]
        best_results_info[f'class_{class_label}_accuracy_std'] = class_stds[class_label]

    return class_acc_dict, by_class_acc_dict, f1_score_dict, best_results_info

In [11]:
input_dim = 3  # Number of node features; BZR dataset has 3D node attributes
hidden_dim = 64  # defines the number of neurons in the GCN layers

# Set up for multi-class classification (family labels)
# Create mapping from string family labels to numeric indices
unique_family_labels = sorted(list(set(filtered_family_labels)))
family_label_to_idx = {label: idx for idx, label in enumerate(unique_family_labels)}
print(f"Family label mapping: {family_label_to_idx}")

# Convert NetworkX graphs to PyTorch Geometric Data objects for multi-class
full_graph_data_multi = []
for idx in valid_indices:
    graph_id = list(graphs.keys())[idx]
    nx_graph = graphs[graph_id]
    family_label_str = family_labels[idx]
    family_label_numeric = family_label_to_idx[family_label_str]
    pyg_data = networkx_to_pyg(nx_graph, family_label_numeric, input_dim)
    full_graph_data_multi.append(pyg_data)

print("BZR multi-class dataset processed successfully.")

# Convert filtered family labels to numeric indices for GCN
filtered_family_labels_numeric = [family_label_to_idx[label] for label in filtered_family_labels]

# Run GCN for multi-class classification
print("\n" + "=" * 60)
print("GCN MULTI-CLASS CLASSIFICATION")
print("=" * 60)
class_acc_multi, by_class_acc_multi, f1_score_multi, results_multi = gcn_multi_class_classification(
    full_graph_data_multi, filtered_family_labels_numeric, input_dim, hidden_dim, num_folds=5, seed=42
)

[I 2025-11-10 20:48:33,633] A new study created in memory with name: no-name-10dedaed-8048-4308-bb1d-03bc558ac156


Family label mapping: {'A.1': 0, 'A.13': 1, 'A.15': 2, 'A.24': 3, 'A.26': 4, 'E.2': 5, 'I': 6, 'M.1': 7, 'M.5': 8, 'N.1': 9}
BZR multi-class dataset processed successfully.

GCN MULTI-CLASS CLASSIFICATION
Using device: cpu
Multi-class classification with 10 classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Starting Bayesian Optimization for Hyperparameter Tuning...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader


BAYESIAN OPTIMIZATION RESULTS
Best trial: 23
Best accuracy: 92.09%
Best hyperparameters:
  num_epochs: 90
  learning_rate: 0.007500121819877176
  batch_size: 32

Optimization time: 476.07 seconds

TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS

--- Training Fold 1/5 ---
Epoch 20/90, Loss: 0.5199
Epoch 40/90, Loss: 0.3968
Epoch 60/90, Loss: 0.3241
Epoch 80/90, Loss: 0.2993
Results for Fold 1:
  Overall Accuracy: 91.53%
  F1 Score (weighted avg): 89.96%
  Class 0 Accuracy: 100.00%
  Class 1 Accuracy: 100.00%
  Class 2 Accuracy: 50.00%
  Class 3 Accuracy: 100.00%
  Class 4 Accuracy: 0.00%
  Class 5 Accuracy: 100.00%
  Class 6 Accuracy: 100.00%
  Class 7 Accuracy: 88.89%
  Class 8 Accuracy: 100.00%
  Class 9 Accuracy: 87.50%

--- Training Fold 2/5 ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


...

Epoch 20/90, Loss: 0.6091
Epoch 40/90, Loss: 0.3835
Epoch 60/90, Loss: 0.3623
Epoch 80/90, Loss: 0.2824
Results for Fold 3:
  Overall Accuracy: 89.66%
  F1 Score (weighted avg): 89.18%
  Class 0 Accuracy: 100.00%
  Class 1 Accuracy: 100.00%
  Class 2 Accuracy: 50.00%
  Class 3 Accuracy: 100.00%
  Class 4 Accuracy: 100.00%
  Class 5 Accuracy: 100.00%
  Class 6 Accuracy: 100.00%
  Class 7 Accuracy: 55.56%
  Class 8 Accuracy: 100.00%
  Class 9 Accuracy: 87.50%

--- Training Fold 4/5 ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 20/90, Loss: 0.5543
Epoch 40/90, Loss: 0.3362
Epoch 60/90, Loss: 0.2911
Epoch 80/90, Loss: 0.2294
Results for Fold 4:
  Overall Accuracy: 89.66%
  F1 Score (weighted avg): 89.55%
  Class 0 Accuracy: 94.44%
  Class 1 Accuracy: 100.00%
  Class 2 Accuracy: 100.00%
  Class 3 Accuracy: 83.33%
  Class 4 Accuracy: 100.00%
  Class 5 Accuracy: 50.00%
  Class 6 Accuracy: 100.00%
  Class 7 Accuracy: 77.78%
  Class 8 Accuracy: 100.00%
  Class 9 Accuracy: 87.50%

--- Training Fold 5/5 ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 20/90, Loss: 0.5501
Epoch 40/90, Loss: 0.4133
Epoch 60/90, Loss: 0.3984
Epoch 80/90, Loss: 0.3473
Results for Fold 5:
  Overall Accuracy: 89.66%
  F1 Score (weighted avg): 89.44%
  Class 0 Accuracy: 100.00%
  Class 1 Accuracy: 100.00%
  Class 2 Accuracy: 50.00%
  Class 3 Accuracy: 100.00%
  Class 4 Accuracy: 100.00%
  Class 5 Accuracy: 100.00%
  Class 6 Accuracy: 83.33%
  Class 7 Accuracy: 66.67%
  Class 8 Accuracy: 66.67%
  Class 9 Accuracy: 100.00%

FINAL GCN MULTI-CLASS RESULTS
Best hyperparameters used:
  Epochs: 90
  Learning Rate: 0.007500
  Batch Size: 32

Final Cross-Validation Results:
Average Overall Accuracy: 90.72% ± 1.40%
Average F1 Score (weighted avg): 90.20% ± 1.35%
Average Class 0 Accuracy: 98.89% ± 2.22%
Average Class 1 Accuracy: 100.00% ± 0.00%
Average Class 2 Accuracy: 70.00% ± 24.49%
Average Class 3 Accuracy: 96.67% ± 6.67%
Average Class 4 Accuracy: 70.00% ± 40.00%
Average Class 5 Accuracy: 90.00% ± 20.00%
Average Class 6 Accuracy: 96.67% ± 6.67%
Average Clas